In [ ]:
from astropy.io import fits 
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd 
from typing import Literal

In [ ]:
with fits.open('/home/bekah/m3-pipeline-dev/data/moon_dss_ghost_corr.fits') as hdul:
    data = hdul[0].data

# with fits.open('/home/bekah/m3-pipeline-dev/data/l1b/m3g20090813t212753_l1b_rdn.fits') as hdul:
#     data = hdul[0].data

In [ ]:
data.shape

In [ ]:
# gaussian filters 

import numpy as np
from scipy.ndimage import gaussian_filter1d

sigma = 70  
illumination = gaussian_filter1d(data[3,:,:], sigma=sigma, axis=1)

eps = 1e-6
corrected = data[3,:,:] / np.maximum(illumination, eps)

corrected *= np.mean(illumination)

fits.writeto('mean_gauss.fits', result, overwrite=True)

In [ ]:
# gaussian filter 

img = data[3,:,:]

print(img.shape)

background = gaussian_filter1d(img, sigma=80, axis=1)   
corrected = img - background

fits.writeto('mean_gauss.fits', corrected, overwrite=True)

In [ ]:
# gaussian filter plus normalization 

from scipy.ndimage import gaussian_filter1d

img = data[3, :, :]

sigma = 6
scatter_fraction = 0.2   
smoothed = gaussian_filter1d(img, sigma=sigma, axis=1)  
background = scatter_fraction * smoothed

corrected = (img - background) * 1.25

fits.writeto('gauss_filter.fits', corrected, overwrite=True)

In [ ]:
# median filter

from scipy.ndimage import median_filter

img = data[3, :, :]

background = median_filter(img, size=(1, 15)) 
corrected = img - background

fits.writeto('median_filter.fits', corrected, overwrite=True)

In [ ]:
# richardson lucy 

import numpy as np
from scipy.signal import fftconvolve

def make_psf_row(width, sigma, alpha, core_frac=None):
    x = np.arange(width) - width // 2
    gauss = np.exp(-(x**2) / (2 * sigma**2))
    gauss /= gauss.sum()

    psf = alpha * gauss
    psf[width // 2] += (1 - alpha)
    psf /= psf.sum()
    return psf.astype(np.float32)


def richardson_lucy_row(row, psf, iterations=25, eps=1e-6):
    estimate = row.copy().astype(np.float32)
    psf_mirror = psf[::-1]

    for _ in range(iterations):
        conv = fftconvolve(estimate, psf, mode='same')
        conv[conv < eps] = eps
        relative_blur = row / conv
        correction = fftconvolve(relative_blur, psf_mirror, mode='same')
        estimate *= correction

    return estimate


def deconvolve_scattered_light(img, sigma=30, alpha=0.4, iterations=25):
    
    img = img.astype(np.float32)
    rows, cols = img.shape

    psf = make_psf_row(cols, sigma=sigma, alpha=alpha)

    corrected = np.zeros_like(img)
    for r in range(rows):
        corrected[r] = richardson_lucy_row(img[r], psf, iterations=iterations)

    return corrected


corrected = deconvolve_scattered_light(data[3,:,:], sigma=30, alpha=0.1, iterations=10)
fits.writeto('rl_corrected.fits', corrected, overwrite=True)

In [ ]:
# wiener deconvolution 

def make_psf_row(width, sigma, alpha):
    """PSF = (1-alpha) delta at center + alpha * Gaussian scatter, normalized to sum=1."""
    x = np.arange(width) - width // 2
    gauss = np.exp(-(x**2) / (2 * sigma**2))
    gauss /= gauss.sum()

    psf = alpha * gauss
    psf[width // 2] += (1 - alpha)
    psf /= psf.sum()

    psf = np.fft.ifftshift(psf)
    return psf.astype(np.float64)


def wiener_deconvolve_2d(img, sigma=30, alpha=0.4, K=0.01, pad_mode='reflect'):

    img = img.astype(np.float64)
    rows, cols = img.shape

    pad = cols // 2
    padded = np.pad(img, ((0, 0), (pad, pad)), mode=pad_mode)
    pcols = padded.shape[1]

    psf = make_psf_row(pcols, sigma=sigma, alpha=alpha)
    psf_fft = np.fft.rfft(psf)

    img_fft = np.fft.rfft(padded, axis=1)

    psf_conj = np.conj(psf_fft)
    wiener_filter = psf_conj / (psf_fft * psf_conj + K)

    result_fft = img_fft * wiener_filter[None, :]
    result = np.fft.irfft(result_fft, n=pcols, axis=1)

    return result[:, pad:pad + cols]


corrected = wiener_deconvolve_2d(data[3,:,:], sigma=41, alpha=0.15, K=0.01) # k was .01
fits.writeto('wiener_corrected.fits', corrected, overwrite=True)


In [ ]:
# tikhonov 

def make_psf_row(width, sigma, alpha):
    x = np.arange(width) - width // 2
    gauss = np.exp(-(x**2) / (2 * sigma**2))
    gauss /= gauss.sum()

    psf = alpha * gauss
    psf[width // 2] += (1 - alpha)
    psf /= psf.sum()

    psf = np.fft.ifftshift(psf)
    return psf.astype(np.float64)


def tikhonov_deconvolve_2d(img, sigma=30, alpha=0.4, lam=0.05, order=2, pad_mode='reflect'):
    img = img.astype(np.float64)
    rows, cols = img.shape

    pad = cols // 2
    padded = np.pad(img, ((0, 0), (pad, pad)), mode=pad_mode)
    pcols = padded.shape[1]

    psf = make_psf_row(pcols, sigma=sigma, alpha=alpha)
    psf_fft = np.fft.rfft(psf)

    freqs = np.fft.rfftfreq(pcols)
    omega = 2 * np.pi * freqs
    if order == 0:
        D_fft = np.ones_like(omega)                
    elif order == 1:
        D_fft = (1j * omega)                        
    elif order == 2:
        D_fft = -(omega ** 2)                        

    D_mag2 = np.abs(D_fft) ** 2

    img_fft = np.fft.rfft(padded, axis=1)
    psf_conj = np.conj(psf_fft)
    denom = (np.abs(psf_fft) ** 2 + lam * D_mag2)
    tik_filter = psf_conj / denom

    result_fft = img_fft * tik_filter[None, :]
    result = np.fft.irfft(result_fft, n=pcols, axis=1)

    return result[:, pad:pad + cols]


corrected = tikhonov_deconvolve_2d(data[3,:,:], sigma=50, alpha=0.2, lam=0.1, order=2)
fits.writeto('tikh_corrected.fits', corrected, overwrite=True)

